# Reasoning Agent

RuleTaker uses forward chaining, CounterBench uses a Boolean causal model, and FOLIO uses first-order logic with Z3. Natural-language translation is separate from the reasoning step.


In [1]:
import csv
import json
import time
from pathlib import Path

import pandas as pd


In [2]:

from dataclasses import dataclass
from itertools import product
from collections import defaultdict
import json

class LogicError(ValueError): pass

def walk_validate(f, bound=None):
    bound = set(bound or [])
    if not isinstance(f, dict): raise LogicError('Formula must be an object')
    op=f.get('op')
    if op=='atom':
        if not isinstance(f.get('pred'),str) or not f['pred'].strip():raise LogicError('Invalid predicate')
        args=f.get('args',[])
        if not isinstance(args,list) or not all(isinstance(x,str) for x in args):raise LogicError('Atom arguments must be strings')
        for x in args:
            if x.startswith('?') and x not in bound:raise LogicError(f'Unbound variable {x}')
    elif op=='not':walk_validate(f['arg'],bound)
    elif op in ('and','or'):
        args=f.get('args',[])
        if not isinstance(args,list) or not args:raise LogicError(f'{op} needs nonempty args')
        for a in args:walk_validate(a,bound)
    elif op in ('implies','iff'):
        walk_validate(f['left'],bound);walk_validate(f['right'],bound)
    elif op in ('forall','exists'):
        var=f.get('var')
        if not isinstance(var,str) or not var.startswith('?'):raise LogicError('Quantified variable must start with ?')
        walk_validate(f['body'],bound|{var})
    else:raise LogicError(f'Unsupported logical operation {op}')

def validate_ast_translation(data):
    if not isinstance(data,dict):raise LogicError('Translation must be JSON object')
    premises=data.get('premises')
    if not isinstance(premises,list):raise LogicError('Expected premises list')
    if 'query' not in data:raise LogicError('Missing query')
    for x in premises+[data['query']]:walk_validate(x)
    # predicate signature consistency catches e.g. like(x) vs like(x,y)
    arities={}
    def scan(f):
        op=f['op']
        if op=='atom':
            name=f['pred']; arity=len(f.get('args',[]))
            if name in arities and arities[name]!=arity:raise LogicError(f'Inconsistent arity for {name}')
            arities[name]=arity
        elif op=='not':scan(f['arg'])
        elif op in ('and','or'):
            for x in f['args']:scan(x)
        elif op in ('implies','iff'):scan(f['left']);scan(f['right'])
        else:scan(f['body'])
    for x in premises+[data['query']]:scan(x)
    return data

def A(pred,*args):return {'op':'atom','pred':pred,'args':list(args)}
def NOT(f):return {'op':'not','arg':f}
def AND(*items):return {'op':'and','args':list(items)}
def OR(*items):return {'op':'or','args':list(items)}
def IMP(a,b):return {'op':'implies','left':a,'right':b}
def ALL(var,body):return {'op':'forall','var':var,'body':body}
def EX(var,body):return {'op':'exists','var':var,'body':body}

def collect_constants(node):
    if isinstance(node,list):
        out=set()
        for x in node:out|=collect_constants(x)
        return out
    if not isinstance(node,dict):return set()
    if node.get('op')=='atom':return {x for x in node.get('args',[]) if not x.startswith('?')}
    out=set()
    for value in node.values():
        if isinstance(value,(list,dict)):out|=collect_constants(value)
    return out

def ground(f, domain, env=None, budget=None):
    env=env or {}; op=f['op']
    if budget is not None:
        budget[0]-=1
        if budget[0]<0:raise LogicError('Grounding budget exceeded')
    if op=='atom':return ('atom',f['pred'],tuple(env.get(x,x) for x in f.get('args',[])))
    if op=='not':return ('not',ground(f['arg'],domain,env,budget))
    if op in ('and','or'):return (op,tuple(ground(x,domain,env,budget) for x in f['args']))
    if op in ('implies','iff'):return (op,ground(f['left'],domain,env,budget),ground(f['right'],domain,env,budget))
    if op in ('forall','exists'):
        parts=[]
        for item in domain:
            e=dict(env);e[f['var']]=item
            parts.append(ground(f['body'],domain,e,budget))
        return ('and' if op=='forall' else 'or',tuple(parts))
    raise LogicError(op)

class CNFEncoder:
    """Tseitin equivalence encoding for an arbitrary grounded Boolean AST."""
    def __init__(self):self.vars={};self.clauses=[];self.next_id=1
    def fresh(self):
        x=self.next_id;self.next_id+=1;return x
    def encode(self,f):
        op=f[0]
        if op=='atom':
            if f not in self.vars:self.vars[f]=self.fresh()
            return self.vars[f]
        if op=='not':return -self.encode(f[1])
        if op=='implies':return self.encode(('or',(('not',f[1]),f[2])))
        if op=='iff':return self.encode(('and',(('implies',f[1],f[2]),('implies',f[2],f[1]))))
        children=[self.encode(x) for x in f[1]]
        v=self.fresh()
        if op=='and':
            for a in children:self.clauses.append([-v,a])
            self.clauses.append([v]+[-a for a in children])
        elif op=='or':
            for a in children:self.clauses.append([v,-a])
            self.clauses.append([-v]+children)
        else:raise LogicError(op)
        return v
    def assert_formula(self,f):self.clauses.append([self.encode(f)])

def dpll(clauses,max_nodes=150000):
    """SAT via deterministic DPLL, unit propagation, pure literals and branching."""
    visited=[0]
    def solve(cnf,assignment):
        visited[0]+=1
        if visited[0]>max_nodes:raise LogicError('DPLL search budget exceeded')
        while True:
            new=[]
            for clause in cnf:
                if any((abs(v) in assignment and assignment[abs(v)]==(v>0)) for v in clause):continue
                rem=[v for v in clause if abs(v) not in assignment]
                if not rem:return None
                new.append(rem)
            cnf=new
            if not cnf:return assignment
            units=[c[0] for c in cnf if len(c)==1]
            if units:
                for u in units:
                    old=assignment.get(abs(u))
                    if old is not None and old!=(u>0):return None
                    assignment[abs(u)]=u>0
                continue
            polar=defaultdict(set)
            for c in cnf:
                for lit in c:polar[abs(lit)].add(lit>0)
            pure=[(v,next(iter(signs))) for v,signs in polar.items() if len(signs)==1]
            if pure:
                assignment.update(pure);continue
            break
        freq=defaultdict(int)
        for c in cnf:
            for lit in c:freq[abs(lit)]+=1
        choice=max(freq,key=freq.get)
        for val in (True,False):
            trial=solve(cnf,{**assignment,choice:val})
            if trial is not None:return trial
        return None
    result=solve([list(x) for x in clauses],{})
    return result is not None,visited[0]

@dataclass
class InferenceResult:
    label: str
    entailed: bool
    contradicted: bool
    supported: bool
    trace: list
    reason: str=''
    sat_calls: int=0

def entailment(data,max_ground_nodes=20000,max_dpll_nodes=150000,max_constants=24):
    """Classical entailment KB |= Q iff KB ∧ ¬Q is UNSAT.
    For an open-world three-valued diagnostic, also check KB |= ¬Q.
    Quantifiers use finite Herbrand-domain grounding: incomplete for general FOL.
    """
    validate_ast_translation(data)
    kb=data['premises'];query=data['query']
    domain=sorted(collect_constants(kb+[query])) or ['_domain_element']
    if len(domain)>max_constants:raise LogicError('Domain limit exceeded')
    budget=[max_ground_nodes]
    grounds=[ground(x,domain,budget=budget) for x in kb]
    q=ground(query,domain,budget=budget)
    def sat(extra):
        enc=CNFEncoder()
        for f in grounds:enc.assert_formula(f)
        enc.assert_formula(extra)
        return dpll(enc.clauses,max_nodes=max_dpll_nodes)
    # Query refutation; KB inconsistency must not silently count as a proof.
    enc=CNFEncoder()
    for f in grounds:enc.assert_formula(f)
    kb_sat,_=dpll(enc.clauses,max_nodes=max_dpll_nodes)
    if not kb_sat:raise LogicError('Knowledge base inconsistent; classical entailment explosive')
    sat_not_q,n1=sat(('not',q))
    sat_q,n2=sat(q)
    entailed=not sat_not_q;contradicted=not sat_q
    if entailed:label='True'
    elif contradicted:label='False'
    else:label='Unknown'
    trace=[f'Finite domain: {domain}',f'KB satisfiable: {kb_sat}',f'SAT(KB AND NOT query): {sat_not_q}',f'SAT(KB AND query): {sat_q}',f'Classical refutation result: {label}']
    return InferenceResult(label,entailed,contradicted,True,trace,
            'Finite-domain first-order grounding + Tseitin CNF + own DPLL',3)

def causal_inference(data):
    """Boolean structural causal model with acyclic equations and do-intervention.
    Fields: variables list; equations {node: formula with atom(pred=variable,args=[])};
    exogenous {variable: bool}; intervention {variable: bool}; target formula.
    """
    variables=data.get('variables',[])
    if not variables or not isinstance(variables,list):raise LogicError('Missing causal variables')
    eqs=data.get('equations',{}); exo=data.get('exogenous',{}); intervention=data.get('intervention',{})
    if not isinstance(eqs,dict) or not isinstance(exo,dict) or not isinstance(intervention,dict):raise LogicError('Malformed causal model')
    known=set(variables)
    for k in list(eqs)+list(exo)+list(intervention):
        if k not in known:raise LogicError(f'Unknown causal variable {k}')
    for value in list(exo.values())+list(intervention.values()):
        if not isinstance(value,bool):raise LogicError('Causal assignments must be booleans')
    def evaluate_formula(f,env):
        op=f['op']
        if op=='atom':
            if f.get('args'):raise LogicError('Causal atom must be zero-arity')
            return resolve(f['pred'])
        if op=='not':return not evaluate_formula(f['arg'],env)
        if op=='and':return all(evaluate_formula(x,env) for x in f['args'])
        if op=='or':return any(evaluate_formula(x,env) for x in f['args'])
        if op=='implies':return (not evaluate_formula(f['left'],env)) or evaluate_formula(f['right'],env)
        raise LogicError('Unsupported causal operator '+str(op))
    env={};stack=set()
    def resolve(v):
        if v in env:return env[v]
        if v not in known:raise LogicError(f'Unknown variable {v}')
        if v in stack:raise LogicError('Cyclic causal model')
        stack.add(v)
        try:
            if v in intervention:val=intervention[v]
            elif v in exo:val=exo[v]
            elif v in eqs:val=evaluate_formula(eqs[v],env)
            else:raise LogicError(f'Unspecified root variable {v}; do not guess')
            env[v]=val;return val
        finally:stack.remove(v)
    truth=evaluate_formula(data['target'],env)
    return InferenceResult('True' if truth else 'False',truth,not truth,True,
        [f'do({intervention})',f'Calculated variables: {dict(sorted(env.items()))}',f'Target={truth}'],
        'Boolean SCM intervention with topological recursive evaluation')


In [3]:

from dataclasses import dataclass
from collections import defaultdict
from itertools import product

@dataclass(frozen=True)
class Lit:
    pred: str
    args: tuple
    neg: bool = False
    @property
    def atom(self): return (self.pred, self.args)

@dataclass(frozen=True)
class Rule:
    body: tuple
    head: Lit

@dataclass
class RuleResult:
    label: str
    entailed: bool
    contradicted: bool
    supported: bool
    trace: list
    reason: str = 'Stratified closed-world rule inference'

def literal(node):
    if node.get('op') == 'not':
        child=node.get('arg',{})
        if child.get('op')!='atom':
            raise LogicError('Only atomic negation is supported in RuleTaker mode')
        return Lit(child['pred'],tuple(child['args']),True)
    if node.get('op') == 'atom':
        return Lit(node['pred'],tuple(node['args']))
    raise LogicError('Expected atomic literal')

def conjunction(node):
    if node.get('op')=='and':
        return tuple(x for child in node['args'] for x in conjunction(child))
    return (literal(node),)

def extract_rule(node):
    while node.get('op')=='forall':
        node=node['body']
    if node.get('op')=='implies':
        return Rule(conjunction(node['left']),literal(node['right']))
    return Rule((),literal(node))

def signature(l):return (l.pred,len(l.args))

def compile_program(data):
    validate_ast_translation(data)
    rules=[extract_rule(f) for f in data['premises']]
    for rule in rules:
        headvars={x for x in rule.head.args if x.startswith('?')}
        posvars={x for lit in rule.body if not lit.neg for x in lit.args if x.startswith('?')}
        # Range-restricted rules are required for safe closed-world negation.
        # A rule such as 'if something is not red then it is blue' can be
        # grounded over the explicit finite domain, so permit its variables.
        bodyvars={x for lit in rule.body for x in lit.args if x.startswith('?')}
        if not headvars <= bodyvars:
            raise LogicError('Rule head contains variable absent from its body')
    return rules

def strata_for(rules):
    """Stratify *ground signed literals*, not predicate names.

    This matters for 'not chase(lion,dog) -> chase(lion,cow)': it is
    acyclic at ground-atom level though both use predicate 'chase'.
    """
    nodes={(r.head.atom,r.head.neg) for r in rules}
    edges=[]
    for r in rules:
        head=(r.head.atom,r.head.neg)
        for lit in r.body:
            # Negative-as-failure tests depend on the *positive* atom.
            dep=(lit.atom,False)
            nodes.add(dep)
            edges.append((dep,head,int(lit.neg)))
    levels={key:0 for key in nodes}
    for _ in range(len(nodes)):
        updated=False
        for dep,head,weight in edges:
            candidate=levels[dep]+weight
            if levels[head]<candidate:
                levels[head]=candidate
                updated=True
        if not updated:break
    else:
        raise LogicError('Negative dependency cycle: program not stratifiable')
    return levels

def ground_literal(lit,bindings):
    return Lit(lit.pred,tuple(bindings.get(a,a) for a in lit.args),lit.neg)

def infer_closed_world(data,max_ground_rules=50000,max_derived=50000):
    rules=compile_program(data)
    query=literal(data['query'])
    constants=sorted(collect_constants(data['premises']+[data['query']]))
    domain=constants or ['_domain_element']
    grounded=[]
    count=0
    for rule in rules:
        vars_=sorted({x for lit in rule.body+(rule.head,) for x in lit.args if x.startswith('?')})
        num=len(domain)**len(vars_)
        if count+num>max_ground_rules:
            raise LogicError('Ground rule budget exceeded')
        count+=num
        for values in product(domain,repeat=len(vars_)):
            env=dict(zip(vars_,values))
            g=Rule(tuple(ground_literal(x,env) for x in rule.body),ground_literal(rule.head,env))
            grounded.append(g)
    strata=strata_for(grounded)
    by_level=defaultdict(list)
    for g in grounded:
        by_level[strata[(g.head.atom,g.head.neg)]].append(g)
    positive=set();explicit_negative=set();proof={}
    # At stratum k, negated body atoms only inspect lower strata; because
    # dependencies are strict, these positive relations are already complete.
    for level in sorted(by_level):
        changed=True
        while changed:
            changed=False
            for rule in by_level[level]:
                ok=all((b.atom not in positive if b.neg else b.atom in positive)
                       for b in rule.body)
                if not ok:continue
                target=explicit_negative if rule.head.neg else positive
                if rule.head.atom in target:continue
                target.add(rule.head.atom)
                proof[(rule.head.atom,rule.head.neg)]=rule
                if len(positive)+len(explicit_negative)>max_derived:
                    raise LogicError('Derivation budget exceeded')
                changed=True
    conflicts=positive & explicit_negative
    if conflicts:
        raise LogicError(f'Contradictory positive and explicit negative facts: {list(conflicts)[:3]}')
    # Under closed-world semantics, NOT P succeeds exactly when P is absent.
    yes=(query.atom not in positive) if query.neg else (query.atom in positive)
    label='True' if yes else 'False'
    trace=[f'Entities: {domain}',f'Ground rules: {count}',f'Strata: {len(set(strata.values()))}',
           f'Derived positive facts: {len(positive)}',f'Derived explicit negatives: {len(explicit_negative)}',
           f'Query: {"not " if query.neg else ""}{query.pred}{query.args}',
           f'Closed-world result: {label}']
    if not query.neg and yes:
        def explain(atom,depth=0,seen=None):
            seen=set() if seen is None else seen
            if atom in seen or depth>12:return []
            seen.add(atom)
            rule=proof.get((atom,False))
            if rule is None:return []
            head=f'{rule.head.pred}{rule.head.args}'
            if not rule.body:return [f'{head} <= given fact']
            lines=[f'{head} <= '+', '.join(('NAF ' if b.neg else '')+f'{b.pred}{b.args}' for b in rule.body)]
            for b in rule.body:
                if not b.neg:lines+=explain(b.atom,depth+1,seen)
            return lines
        trace+=explain(query.atom)
    return RuleResult(label,yes,not yes,True,trace)


In [4]:

import hashlib
import json
import re
from pathlib import Path

LOGIC_PROMPT = '''You are a compiler converting natural-language premises and a query into logical syntax.
You MUST NOT infer or output the truth of the query. Do not answer the query.
Return exactly ONE JSON object with keys "premises" (array) and "query" (formula).
Permitted formula forms:
{"op":"atom","pred":"predicate_name","args":["bob","?x"]}
{"op":"not","arg":FORMULA}
{"op":"and","args":[FORMULA,FORMULA]}
{"op":"or","args":[FORMULA,FORMULA]}
{"op":"implies","left":FORMULA,"right":FORMULA}
{"op":"iff","left":FORMULA,"right":FORMULA}
{"op":"forall","var":"?x","body":FORMULA}
{"op":"exists","var":"?x","body":FORMULA}
Use identical predicates for identical relations across premises and query.
For e.g. "Bob is young" use {"op":"atom","pred":"young","args":["bob"]}.
For "If someone is young then they are smart" use a forall ?x containing implies(young(?x), smart(?x)).
For "someone likes the tiger" use a two-argument like(?x,tiger) predicate.
Preserve negation, conjunction, disjunction, and quantifier scope faithfully.
If meaning cannot be represented faithfully, return {"error":"untranslatable"}.
Output JSON only, with no markdown fences or commentary.'''

CAUSAL_PROMPT = '''You are a compiler converting a natural language counterfactual scenario into a Boolean causal model.
You MUST NOT answer the question or output a prediction. Return JSON only.
Structure: {"variables":["a","b"],"equations":{"b":FORMULA},
"exogenous":{"a":true},"intervention":{"a":false},"target":FORMULA}
FORMULA is an AST of zero-argument atoms, not, and, or (same field names as logic).
Only assume deterministic causal equations or initial states when explicitly warranted.
If unspecified, output {"error":"underspecified causal model"}.
Do not turn mere causal associations into deterministic equations.'''


def parse_json_text(text):
    if not isinstance(text, str) or not text.strip():
        raise LogicError('LLM returned empty text')
    s = text.strip()
    s = re.sub(r'^```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```$', '', s).strip()
    try:
        obj = json.loads(s)
    except json.JSONDecodeError:
        start = s.find('{')
        if start < 0:
            raise LogicError('LLM returned no JSON object. Preview: '+repr(s[:250]))
        try:
            obj, _ = json.JSONDecoder().raw_decode(s[start:])
        except json.JSONDecodeError as e:
            raise LogicError('LLM returned malformed or truncated JSON: '+repr(s[:350])) from e
    if not isinstance(obj, dict):
        raise LogicError('LLM returned JSON but not a JSON object')
    return obj


def _validate_causal(data):
    if not isinstance(data.get('variables'),list) or not data['variables']:
        raise LogicError('Causal translation missing variables')
    if not isinstance(data.get('equations'),dict) or not isinstance(data.get('exogenous'),dict):
        raise LogicError('Causal translation missing equations/exogenous')
    if not isinstance(data.get('intervention'),dict) or 'target' not in data:
        raise LogicError('Causal translation missing intervention/target')
    return data



# Deterministic structural normalization: only universally close conditional rules
# containing generic variables. Never auto-close existential statements or queries.
def _free_vars(formula, bound=None):
    bound=set() if bound is None else set(bound)
    op=formula.get('op')
    if op=='atom':
        return {x for x in formula.get('args',[]) if isinstance(x,str) and x.startswith('?') and x not in bound}
    if op in ('forall','exists'):
        return _free_vars(formula['body'],bound|{formula['var']})
    if op=='not':return _free_vars(formula['arg'],bound)
    if op in ('and','or'):
        z=set()
        for arg in formula['args']:z|=_free_vars(arg,bound)
        return z
    if op in ('implies','iff'):
        return _free_vars(formula['left'],bound)|_free_vars(formula['right'],bound)
    return set()

def close_universal_rule(formula, sentence):
    if not isinstance(formula,dict):raise LogicError('Formula must be an object')
    # Only safe when natural language has a universal conditional form.
    if (formula.get('op')=='implies' and
        re.match(r'^\s*if\s+.+\bthen\b', sentence, re.I)):
        for var in sorted(_free_vars(formula),reverse=True):
            formula={'op':'forall','var':var,'body':formula}
    return formula

def split_statements(text):
    if isinstance(text,list):return [str(x).strip() for x in text if str(x).strip()]
    # RuleTaker statements are sentence-separated, FOLIO sometimes newline-separated.
    return [x.strip() for x in re.split(r'(?<=[.!?])\s+|\n+',str(text).strip()) if x.strip()]

VERBS={'likes':'like','like':'like','needs':'need','need':'need','chases':'chase','chase':'chase',
       'sees':'see','see':'see','visits':'visit','visit':'visit','eats':'eat','eat':'eat'}

def fast_parse(sentence):
    """Conservative deterministic parser for explicit simple RuleTaker grammar.
    Any uncertain structure goes to the LLM, not to guessed logic.
    """

    s=str(sentence).strip().rstrip('.!?').lower().strip()
    if not s:return None
    var_marker=bool(re.search(r'\b(someone|something|they|it)\b',s))
    # RuleTaker categorical universals:
    # "All blue people are young" => forall x. blue(x) -> young(x)
    # "Furry, smart people are big" => forall x. furry(x) & smart(x) -> big(x)
    # "All red people are not smart" => forall x. red(x) -> not smart(x)
    m=re.fullmatch(
        r'(?:all\s+)?([a-z]+(?:\s*,\s*[a-z]+)*(?:\s+and\s+[a-z]+)?)\s+'
        r'(people|things|animals)\s+(?:are|is)\s+(not\s+)?([a-z]+)',
        s
    )
    if m:
        adjectives, _category, neg, outcome=m.groups()
        # Explicit modifier conjunction; no extra entity-type predicate is assumed
        # for this RuleTaker template. Extend separately if dataset needs it.
        modifiers=[p.strip() for p in re.split(r'\s*,\s*|\s+and\s+',adjectives)]
        body_atoms=[A(p,'?x') for p in modifiers]
        body=body_atoms[0] if len(body_atoms)==1 else AND(*body_atoms)
        head=A(outcome,'?x')
        if neg: head=NOT(head)
        return ALL('?x',IMP(body,head))
    m=re.fullmatch(r'if (.+) then (.+)',s)
    if m:
        left,right=m.groups()
        # Conjunctive condition, including omission of a repeated subject:
        # "someone is smart and kind" -> same subject twice
        pieces=left.split(' and ')
        if len(pieces)>1 and re.fullmatch(r'(?:someone|something|they|it|[a-z]+) is (?:not )?[a-z]+',pieces[0]):
            subject=pieces[0].split(' is ')[0]
            pieces=[pieces[0]]+[(subject+' is '+p if re.fullmatch(r'(?:not )?[a-z]+',p) else p) for p in pieces[1:]]
        body=[_simple_atom(p) for p in pieces]
        head=_simple_atom(right)
        if any(x is None for x in body) or head is None:return None
        condition=body[0] if len(body)==1 else AND(*body)
        f=IMP(condition,head)
        if var_marker or _free_vars(f):
            return close_universal_rule(f,s)
        return f
    return _simple_atom(s)

def _simple_atom(s):

    s=s.strip().lower().rstrip('.')
    # adjective facts: "bob is not young" and "they are smart"
    m=re.fullmatch(r'(?:the )?([a-z]+(?: [a-z]+)?) (?:is|are) (not )?([a-z]+)',s)
    if m:
        subject,neg,pred=m.groups()
        subject='?x' if subject in ('someone','something','they','it') else subject.replace(' ','_')
        f=A(pred,subject)
        return NOT(f) if neg else f
    # Use an explicit known verb boundary, NOT a greedy subject/object regex.
    # Previous regex parsed "the cat chases the cow" incorrectly as
    # subject="cat chases", verb="the", object="cow".
    # Match the longest verb variants first, including explicit negation.
    verb_pattern = '|'.join(sorted(map(re.escape, VERBS), key=len, reverse=True))
    m=re.fullmatch(
        rf'(.+?)\s+(does\s+not\s+|do\s+not\s+)?({verb_pattern})\s+(.+)',
        s
    )
    if m:
        subject, neg, verb, obj = m.groups()
        def entity(term):
            term = re.sub(r'^the\s+', '', term.strip())
            if not re.fullmatch(r'[a-z]+(?:\s+[a-z]+)*', term):
                return None
            return '?x' if term in ('someone','something','they','it') else term.replace(' ','_')
        subject, obj = entity(subject), entity(obj)
        if subject is None or obj is None:
            return None
        f=A(VERBS[verb],subject,obj)
        return NOT(f) if neg else f
    return None

def predicate_signatures(formulas):
    found={}
    def visit(x):
        if not isinstance(x,dict):return
        if x.get('op')=='atom':found[x['pred']]=len(x.get('args',[]))
        else:
            for value in x.values():
                if isinstance(value,dict):visit(value)
                elif isinstance(value,list):
                    for y in value:visit(y)
    for f in formulas:visit(f)
    return found

class LLMTranslator:
    def __init__(self, model='Qwen/Qwen2.5-3B-Instruct', cache_dir='.logic_cache_v3',
                 max_new_tokens=1024, device='auto', precision='fp32',
                 trust_remote_code=False, smoke_test=True):
        """precision='fp32' is more memory hungry but safest for first validation.

        If GPU memory is insufficient, choose a smaller model before changing
        precision. You may later benchmark fp16/bf16 once smoke_test passes.
        """
        if device not in ('auto','cpu','cuda'):
            raise ValueError('device must be auto, cpu or cuda')
        if precision not in ('fp32','fp16','bf16'):
            raise ValueError('precision must be fp32, fp16 or bf16')
        self.model = model
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.max_new_tokens = max_new_tokens
        self.device = device
        self.precision = precision
        self.trust_remote_code = trust_remote_code
        self.smoke_test = smoke_test
        self._model = None
        self._tokenizer = None
        self._device = None

    def _ensure_model(self):
        if self._model is not None:
            return
        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
        except ImportError as e:
            raise RuntimeError('Missing torch/transformers. Install inside the active notebook kernel.') from e

        device = ('cuda' if torch.cuda.is_available() else 'cpu') if self.device=='auto' else self.device
        if device=='cuda' and not torch.cuda.is_available():
            raise LogicError('CUDA requested but unavailable in this Python kernel')
        if device=='cpu' and self.precision!='fp32':
            raise LogicError('CPU execution requires precision="fp32" for this implementation')
        if self.precision=='bf16' and device=='cuda' and not torch.cuda.is_bf16_supported():
            raise LogicError('BF16 not supported by your GPU; use fp32 or fp16')
        dtype = {'fp32':torch.float32,'fp16':torch.float16,'bf16':torch.bfloat16}[self.precision]
        print(f'Loading {self.model} on {device} in {self.precision} (no Ollama)')
        tokenizer = AutoTokenizer.from_pretrained(self.model,trust_remote_code=self.trust_remote_code)
        if not getattr(tokenizer,'chat_template',None):
            raise LogicError('Tokenizer has no chat template; choose an instruct/chat model')
        try:
            model = AutoModelForCausalLM.from_pretrained(
                self.model,torch_dtype=dtype,trust_remote_code=self.trust_remote_code,
                low_cpu_mem_usage=True)
            model.to(device)
            model.eval()
        except torch.cuda.OutOfMemoryError as e:
            if torch.cuda.is_available():torch.cuda.empty_cache()
            raise LogicError('GPU out of memory loading model. Use 1.5B model or free GPU memory.') from e
        self._model=model
        self._tokenizer=tokenizer
        self._device=device
        if self.smoke_test:
            try:
                resp = self._generate([{'role':'user','content':'Reply with exactly the word READY.'}],max_new_tokens=16)
                print('Generation smoke test:',repr(resp))
                if not re.search(r'\bREADY\b',resp,re.I):
                    raise LogicError('Generation health check FAILED: '+repr(resp[:160])+
                      '. Stop evaluation. Check model files, precision, GPU and Transformers version.')
            except Exception:
                self._model=None
                self._tokenizer=None
                raise

    def _generate(self,messages,max_new_tokens=None):
        import torch
        tok=self._tokenizer
        model=self._model
        batch=tok.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
                                      return_tensors='pt',return_dict=True)
        batch={k:v.to(self._device) for k,v in batch.items()}
        length=batch['input_ids'].shape[1]
        limit=getattr(model.config,'max_position_embeddings',32768)
        n=max_new_tokens or self.max_new_tokens
        if length+n>limit:
            raise LogicError(f'Prompt too long: {length} tokens + {n} output > {limit}')
        with torch.inference_mode():
            out=model.generate(**batch, max_new_tokens=n, do_sample=False,
                pad_token_id=tok.pad_token_id or tok.eos_token_id,
                eos_token_id=tok.eos_token_id, use_cache=False)
        completion=out[0,length:]
        text=tok.decode(completion,skip_special_tokens=True).strip()
        if not text:
            raise LogicError('Model generated no text')
        if re.search(r'([!?.])\1{35,}',text) or re.search(r'(.{2,8})\1{16,}',text):
            raise LogicError('Degenerate repeated-token output (model inference is unhealthy): '+repr(text[:140]))
        return text

    def _ask(self,system,user):
        self._ensure_model()
        return self._generate([{'role':'system','content':system},
                               {'role':'user','content':user}])

    def _parse_formula_once(self, sentence, vocabulary=None, attempts=2):
        """Ask the model to translate ONE statement, never the entire KB."""
        vocab_note = ('Existing predicate signatures: ' + json.dumps(vocabulary, ensure_ascii=False)) if vocabulary else ''
        base = (
            'Translate exactly ONE natural-language statement to ONE formula JSON object. '
            'No proof, no truth judgment. Return JSON ONLY.\n'
            'Forms: {"op":"atom","pred":"young","args":["bob"]}; '
            '{"op":"not","arg":FORMULA}; '
            '{"op":"and","args":[FORMULA,FORMULA]}; '
            '{"op":"or","args":[FORMULA,FORMULA]}; '
            '{"op":"implies","left":FORMULA,"right":FORMULA}; '
            '{"op":"forall","var":"?x","body":FORMULA}; '
            '{"op":"exists","var":"?x","body":FORMULA}.\n'
            'Universal rules like "If someone is young then they are smart" MUST be '
            'enclosed in forall ?x. Use same predicate names for identical relations. '
            'Do not introduce facts not in the sentence. '
            'If untranslatable, return {"error":"untranslatable"}.\n' + vocab_note
        )
        request = 'Statement: ' + sentence
        error = ''
        for trial in range(attempts):
            raw = self._ask(base, request if trial==0 else request + '\nPrevious error: '+error+'\nReturn one CORRECT, COMPLETE JSON object.')
            try:
                obj=parse_json_text(raw)
                if obj.get('error'):
                    raise LogicError(str(obj['error']))
                obj = close_universal_rule(obj, sentence)

                walk_validate(obj)
                return obj
            except (LogicError, KeyError, TypeError) as exc:
                error=str(exc)
        raise LogicError('Unable to translate statement: '+repr(sentence[:100])+'; last error: '+error)

    def translate(self,context,query,mode='logic'):
        """Separate semantic parsing from inference. No answer is requested from the model.

        A failed translation raises LogicError; evaluation records it as unsupported.
        """
        if mode not in ('logic','causal'):
            raise ValueError(f'Unknown mode: {mode}')
        if mode=='causal':
            # Do not claim that generic Boolean logic is a causal model.
            system=CAUSAL_PROMPT
            request=json.dumps({'knowledge_base':context,'query':query},ensure_ascii=False)
            key=hashlib.sha256(json.dumps([self.model,self.precision,'causal-v4',system,request]).encode()).hexdigest()
            path=self.cache_dir/(key+'.json')
            obj=json.loads(path.read_text()) if path.exists() else parse_json_text(self._ask(system,request))
            if obj.get('error'):raise LogicError('Causal model unavailable: '+str(obj['error']))
            _validate_causal(obj)
            if not path.exists():path.write_text(json.dumps(obj),encoding='utf8')
            return obj

        statements = split_statements(context)
        if not statements:
            raise LogicError('No knowledge-base statements found')
        key_data=[self.model,self.precision,self.max_new_tokens,'sentence-v4',statements,query]
        key=hashlib.sha256(json.dumps(key_data,ensure_ascii=False).encode()).hexdigest()
        path=self.cache_dir/(key+'.json')
        if path.exists():
            result=json.loads(path.read_text(encoding='utf8'))
            validate_ast_translation(result)
            return result
        premises=[]
        # One statement per generation. Shorter outputs avoid truncated KB JSON.
        # Earlier signatures are included only as hints, not logical assertions.
        for statement in statements:
            formula=fast_parse(statement)
            if formula is None:
                formula=self._parse_formula_once(statement, vocabulary=predicate_signatures(premises))
            premises.append(formula)
        query_formula=fast_parse(query)
        if query_formula is None:
            query_formula=self._parse_formula_once(query,vocabulary=predicate_signatures(premises))
        result={'premises':premises,'query':query_formula}
        validate_ast_translation(result)
        path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf8')
        return result


In [5]:

from __future__ import annotations
import re
from collections import defaultdict, deque
from itertools import product

class UnsupportedCounterBench(ValueError):
    pass

NAME = r"[A-Za-z][A-Za-z0-9_]*"
CLAUSE_START = re.compile(
    rf",\s*(?=(?:and\s+)?(?:not\s+)?{NAME}"
    rf"(?:\s+(?:and|or)\s+(?:not\s+)?{NAME})*"
    rf"\s+(?:together\s+)?causes?\b)", re.I
)

def expression(raw):
    raw = re.sub(r"\s+together\s*$", "", raw.strip().lower())
    raw = raw.strip().strip(",")
    or_parts = re.split(r"\s+or\s+", raw)
    if len(or_parts) > 1:
        return ("or", tuple(expression(x) for x in or_parts))
    and_parts = re.split(r"\s+and\s+", raw)
    if len(and_parts) > 1:
        return ("and", tuple(expression(x) for x in and_parts))
    if raw.startswith("not "):
        return ("not", expression(raw[4:]))
    if not re.fullmatch(NAME, raw):
        raise UnsupportedCounterBench(f"Invalid Boolean term: {raw}")
    return ("var", raw)

def vars_in(expr):
    if expr[0] == "var":
        return {expr[1]}
    if expr[0] == "not":
        return vars_in(expr[1])
    out = set()
    for item in expr[1]:
        out.update(vars_in(item))
    return out

def eval_expr(expr, state):
    if expr[0] == "var":
        return state[expr[1]]
    if expr[0] == "not":
        return not eval_expr(expr[1], state)
    if expr[0] == "and":
        return all(eval_expr(x, state) for x in expr[1])
    if expr[0] == "or":
        return any(eval_expr(x, state) for x in expr[1])
    raise UnsupportedCounterBench(f"Unknown logical operator: {expr[0]}")

def parse_rules(given_info):
    info = re.sub(r"^We know that\s+", "", given_info.strip(), flags=re.I)
    observed = {}
    obs_match = re.search(r"\bWe observed\s+(.+?)\s*\.?\s*$", info, flags=re.I)
    if obs_match:
        raw_obs = obs_match.group(1).strip().rstrip(".")
        info = info[:obs_match.start()].rstrip(" .")
        for phrase in re.split(r",\s*(?:and\s+)?|\s+and\s+", raw_obs, flags=re.I):
            phrase = phrase.strip().lower()
            if not phrase:
                continue
            val = not phrase.startswith("not ")
            name = phrase[4:].strip() if not val else phrase
            if not re.fullmatch(NAME, name):
                raise UnsupportedCounterBench(f"Invalid observation: {phrase}")
            if name in observed and observed[name] != val:
                raise UnsupportedCounterBench(f"Conflicting observation: {name}")
            observed[name] = val
    info = info.rstrip(" .")
    pieces = CLAUSE_START.split(info)
    rules = []
    for piece in pieces:
        piece = re.sub(r"^and\s+", "", piece.strip(), flags=re.I)
        match = re.fullmatch(r"(.+?)\s+causes?\s+(.+)", piece, flags=re.I)
        if not match:
            raise UnsupportedCounterBench(f"Unrecognized causal clause: {piece}")
        lhs = expression(match.group(1))
        rhs = re.sub(r",\s*(?:and\s+)", ", ", match.group(2).strip(), flags=re.I)
        effects = re.split(r",\s*|\s+and\s+", rhs, flags=re.I)
        for item in effects:
            item = item.strip().lower()
            negative = item.startswith("not ")
            effect = item[4:].strip() if negative else item
            if not re.fullmatch(NAME, effect):
                raise UnsupportedCounterBench(f"Invalid effect: {item}")
            rules.append({"condition": lhs, "effect": effect, "negative_effect": negative})
    if not rules:
        raise UnsupportedCounterBench("No causal rules")
    return rules, observed

def parse_query(question, family):
    q = question.strip()
    family = family.lower()
    # A named intervention is interpreted as a forced Boolean value.
    def interventions_from(text):
        text = text.strip().rstrip("?. ")
        text = re.sub(r"^(?:if|assume|suppose|further suppose)\s+", "", text, flags=re.I)
        text = re.sub(r"\s+instead\s+of\s+.+$", "", text, flags=re.I)
        text = re.sub(r",\s*and\s+", " and ", text, flags=re.I)
        values = {}
        for part in re.split(r"\s+and\s+|,\s*", text, flags=re.I):
            part = part.strip().lower()
            if not part:
                continue
            m = re.fullmatch(rf"(not\s+)?({NAME})", part, flags=re.I)
            if not m:
                raise UnsupportedCounterBench(f"Unrecognized intervention: {part}")
            name, value = m.group(2).lower(), not bool(m.group(1))
            if name in values and values[name] != value:
                raise UnsupportedCounterBench(f"Conflicting interventions on {name}")
            values[name] = value
        if not values:
            raise UnsupportedCounterBench("No interventions found")
        return values

    # Standard basic, conditional, and joint templates.
    standard = re.fullmatch(
        rf"Would\s+({NAME})\s+occur\s+if\s+(.+?)\s*\?", q, flags=re.I
    )
    if standard:
        target = standard.group(1).lower()
        interventions = interventions_from(standard.group(2))
        return target, [interventions]

    # Nested template: "Assume not X, and based on this assumption,
    # further suppose not Y. Would Z occur?"
    nested = re.fullmatch(
        rf"(?:Assume|Suppose)\s+(.+?),\s*and\s+based\s+on\s+this\s+assumption,\s*"
        rf"(?:further\s+)?suppose\s+(.+?)\.\s*Would\s+({NAME})\s+occur\s*\?",
        q, flags=re.I
    )
    if nested:
        first = interventions_from(nested.group(1))
        second = interventions_from(nested.group(2))
        return nested.group(3).lower(), [first, second]
    raise UnsupportedCounterBench(f"Unsupported {family} question: {q}")

def compile_model(rules):
    equations = defaultdict(list)
    nodes = set()
    parents = defaultdict(set)
    children = defaultdict(set)
    for r in rules:
        effect = r["effect"]
        dependencies = vars_in(r["condition"])
        nodes.add(effect)
        nodes.update(dependencies)
        equations[effect].append(r)
        for p in dependencies:
            parents[effect].add(p)
            children[p].add(effect)
    return nodes, equations, parents, children

def solve(rules, forced, observations=None, observation_mode="clamp", max_cycle_nodes=16):
    """Solve equations; for cycles enumerate fixed points and require uniqueness.

    observation_mode='clamp' reproduces prior diagnostic baseline (endogenous
    observations are held fixed even under interventions). 'root_only' applies
    observed values only to root variables; neither mode performs SCM abduction.
    """
    observations = observations or {}
    nodes, equations, parents, children = compile_model(rules)
    # Intervening on an unconnected variable has no downstream effect.
    # Retain it as an isolated exogenous node rather than treating it as
    # a parse error. This does not silently introduce causal edges.
    for name in set(forced) - nodes:
        nodes.add(name)
    if not set(observations).issubset(nodes):
        raise UnsupportedCounterBench(f"Unknown observation: {set(observations)-nodes}")
    if observation_mode not in ("clamp", "root_only"):
        raise ValueError("observation_mode must be 'clamp' or 'root_only'")

    pins = {n: v for n, v in observations.items()
            if observation_mode == "clamp" or not equations[n]}
    pins.update(forced)  # intervention wins over observational evidence
    dependencies = {n: set(parents[n]) for n in nodes}
    for n in pins:
        dependencies[n] = set()  # intervened/clamped equation is removed
    indegree = {n: len(dependencies[n]) for n in nodes}
    dependents = defaultdict(set)
    for n, deps in dependencies.items():
        for dep in deps:
            dependents[dep].add(n)
    queue = deque(sorted(n for n in nodes if indegree[n] == 0))
    order = []
    while queue:
        n = queue.popleft()
        order.append(n)
        for child in sorted(dependents[n]):
            indegree[child] -= 1
            if indegree[child] == 0:
                queue.append(child)

    def node_value(n, state):
        if n in pins:
            return pins[n]
        if not equations[n]:
            return True  # documented initial-root convention
        return any(
            not eval_expr(r["condition"], state)
            if r["negative_effect"] else eval_expr(r["condition"], state)
            for r in equations[n]
        )

    if len(order) == len(nodes):
        state = {}
        for n in order:
            state[n] = node_value(n, state)
        return state

    # Cyclic dependencies: enumerate satisfying Boolean fixed points.
    # Small graph only. Never break ties with benchmark labels.
    if len(nodes) > max_cycle_nodes:
        raise UnsupportedCounterBench("Cyclic model exceeds enumeration limit")
    solutions = []
    names = sorted(nodes)
    for values in product((False, True), repeat=len(names)):
        candidate = dict(zip(names, values))
        if all(candidate[n] == node_value(n, candidate) for n in names):
            solutions.append(candidate)
            if len(solutions) > 1:
                break
    if not solutions:
        raise UnsupportedCounterBench("Cyclic model has no Boolean fixed point")
    if len(solutions) > 1:
        raise UnsupportedCounterBench("Cyclic model has multiple Boolean fixed points")
    return solutions[0]

def infer_counterbench(row, observation_mode="clamp"):
    family = row["type"].strip().lower()
    if family not in {"basic", "conditional", "joint", "nested"}:
        raise UnsupportedCounterBench(f"Unsupported category: {family}")
    rules, observations = parse_rules(row["given_info"])
    target, stages = parse_query(row["question"], family)
    nodes, _, _, _ = compile_model(rules)
    if target not in nodes:
        raise UnsupportedCounterBench(f"Unknown query variable: {target}")
    # Nested stages progressively accumulate forced assignments. Later stages
    # override earlier stages. Evaluating the final cumulative intervention
    # is a baseline for nested counterfactual questions, not cross-world logic.
    forced = {}
    stage_states = []
    for stage in stages:
        forced.update(stage)
        state = solve(rules, forced, observations,
                      observation_mode=observation_mode)
        stage_states.append(state)
    return {
        "prediction": "True" if stage_states[-1][target] else "False",
        "type": family, "target": target, "observations": observations,
        "intervention_stages": stages, "states": stage_states[-1],
        "rules": rules, "observation_mode": observation_mode,
        "reasoning_method": "boolean_scm_baseline",
    }


In [6]:

import re
import time
from dataclasses import dataclass
from sympy import Symbol, And, Or, Not, Implies, Equivalent, Xor, true, false
from sympy.logic.inference import satisfiable

class FOLParseError(ValueError):
    pass

TOKEN = re.compile(r'\s*(<->|<=>|->|=>|!=|==|[∀∃¬~!∧∨⊕→↔&|()=.,:]|\b(?:forall|exists|and|or|not|xor|implies|iff|True|False)\b|\d+[A-Za-z_]\w*|[^\W\d]\w*|\d+|\S)', re.I)
ALIASES = {'∀':'forall','∃':'exists','¬':'not','~':'not','!':'not',
           '∧':'and','&':'and','∨':'or','|':'or','⊕':'xor',
           '→':'implies','->':'implies','=>':'implies',
           '↔':'iff','<->':'iff','<=>':'iff', '==':'=', '!=':'!='}
PRECEDENCE = {'iff':1,'implies':2,'xor':3,'or':4,'and':5, }

class Parser:
    def __init__(self, source):
        if not isinstance(source,str) or not source.strip():
            raise FOLParseError('Empty formula')
        self.tokens = [ALIASES.get(x,x) for x in TOKEN.findall(source)]
        self.tokens = [x.lower() if x.lower() in ('forall','exists','and','or','not','xor','implies','iff') else x for x in self.tokens]
        self.i=0
    def peek(self):
        return self.tokens[self.i] if self.i < len(self.tokens) else None
    def pop(self, expected=None):
        x=self.peek()
        if x is None or (expected is not None and x != expected):
            raise FOLParseError(f'Expected {expected or "token"}; got {x} near token {self.i}')
        self.i+=1; return x
    def atom(self):
        t=self.pop()
        if t=='(':
            e=self.expr(0);self.pop(')');return e
        if t=='not': return ('not',self.expr(7))
        if t in ('forall','exists'):
            names=[]
            while True:
                var=self.pop()
                if not var.isidentifier():
                    raise FOLParseError('Invalid bound variable: '+var)
                names.append(var)
                if self.peek()==',':
                    self.pop();continue
                break
            if self.peek() in ('.',':'): self.pop()
            # A parenthesized quantifier body ends at its matching closing parenthesis.
            # Otherwise preserve the existing broad scope of an unparenthesized quantifier.
            body=self.atom() if self.peek()=='(' else self.expr(0)
            for var in reversed(names): body=(t,var,body)
            return body
        if t in ('True','False','true','false'): return ('bool',t.lower()=='true')
        if not (t.isidentifier() or re.fullmatch(r'\d+[A-Za-z_]\w*',t)):
            raise FOLParseError('Invalid atom '+repr(t))
        args=[]
        if self.peek()=='(':
            self.pop('(')
            if self.peek()!=')':
                while True:
                    args.append(self.pop())
                    if not (args[-1].isidentifier() or re.fullmatch(r'\d+[A-Za-z_]\w*',args[-1]) or args[-1].isdecimal()):
                        raise FOLParseError('Invalid term '+str(args[-1]))
                    if self.peek()!=',':break
                    self.pop(',')
            self.pop(')')
        if self.peek() in ('=','!=') and not args:
            operator=self.pop()
            other=self.pop()
            if not (other.isidentifier() or re.fullmatch(r'\d+[A-Za-z_]\w*',other) or other.isdecimal()):
                raise FOLParseError('Invalid equality term '+repr(other))
            return ('eq',t,other,operator=='!=')
        return ('pred',t,tuple(args))
    def expr(self,min_prec=0):
        left=self.atom()
        while self.peek() in PRECEDENCE and PRECEDENCE[self.peek()] >= min_prec:
            op=self.pop();p=PRECEDENCE[op]
            # implication is right associative
            right=self.expr(p if op=='implies' else p+1)
            left=(op,left,right)
        return left
    def parse(self):
        node=self.expr()
        if self.peek() is not None:raise FOLParseError(f'Trailing token {self.peek()} at {self.i}')
        return node

def normalize_parentheses(s):
    """Repair only excess outer closing or missing final parentheses.

    Return (normalized, repair_note). Never change an internally balanced formula.
    """
    s = s.strip()
    depth = 0
    extra_positions = []
    for i, ch in enumerate(s):
        if ch == '(':
            depth += 1
        elif ch == ')':
            if depth == 0:
                extra_positions.append(i)
            else:
                depth -= 1
    if extra_positions:
        # Only discard extra ')' at the very end, optionally separated by spaces.
        first = extra_positions[0]
        if s[first:].strip(' )'):
            return s, None
        s = s[:first].rstrip()
    if depth:
        s += ')' * depth
    note = None
    if extra_positions or depth:
        note = ('removed_extra_final_closers=' + str(len(extra_positions)) +
                ';added_missing_final_closers=' + str(depth))
    return s, note


def parse_formula(s):
    return Parser(s).parse()


def parse_formula_repaired(s):
    normalized, note = normalize_parentheses(s)
    return Parser(normalized).parse(), note

def split_premises(s):
    if isinstance(s,list):return [x.strip() for x in s if str(x).strip()]
    if not isinstance(s,str):return []
    # FOLIO commonly stores one annotated formula per line.
    return [line.strip().rstrip(';') for line in s.splitlines() if line.strip()]

def terms_and_constants(node,bound=frozenset()):
    op=node[0]
    if op=='pred':
        return set(x.casefold() for x in node[2] if x not in bound)
    if op=='eq':
        return {x.casefold() for x in (node[1],node[2]) if x not in bound}
    if op in ('forall','exists'):return terms_and_constants(node[2],bound|{node[1]})
    if op=='not':return terms_and_constants(node[1],bound)
    if op=='bool':return set()
    return terms_and_constants(node[1],bound)|terms_and_constants(node[2],bound)

def check_symbol_signatures(nodes):
    signatures={}
    def visit(n):
        op=n[0]
        if op=='eq':return
        if op=='pred':
            name,arity=n[1],len(n[2])
            if name in signatures and signatures[name]!=arity:
                raise FOLParseError(f'Predicate {name} has incompatible arities')
            signatures[name]=arity
        elif op in ('forall','exists','not'):visit(n[-1])
        elif op!='bool':visit(n[1]);visit(n[2])
    for n in nodes:visit(n)
    return signatures

def ground(node,env,domain,cache):
    op=node[0]
    if op=='bool':return true if node[1] else false
    if op=='eq':
        a=env.get(node[1],node[1].casefold()); b=env.get(node[2],node[2].casefold())
        value=(a!=b) if node[3] else (a==b)
        return true if value else false
    if op=='pred':
        name=node[1]; args=tuple(env.get(t,t.casefold()) for t in node[2]);key=(name,args)
        if key not in cache:cache[key]=Symbol(name+'('+','.join(args)+')')
        return cache[key]
    if op=='not':return Not(ground(node[1],env,domain,cache))
    if op in ('forall','exists'):
        xs=[ground(node[2],{**env,node[1]:d},domain,cache) for d in domain]
        return And(*xs) if op=='forall' else Or(*xs)
    a=ground(node[1],env,domain,cache)
    b=ground(node[2],env,domain,cache)
    return {'and':And,'or':Or,'xor':Xor,'implies':Implies,'iff':Equivalent}[op](a,b)

def solve_folio_logic(row, extra_domain=1, max_ground_atoms=180):
    start=time.perf_counter()
    raw=split_premises(row['premises-FOL'])
    parsed=[parse_formula_repaired(x) for x in raw]
    query, query_repair=parse_formula_repaired(row['conclusion-FOL'])
    formulas=[node for node, note in parsed]
    repairs=[note for node,note in parsed if note]
    if query_repair: repairs.append(query_repair)
    check_symbol_signatures(formulas+[query])
    constants=set()
    for x in formulas+[query]:constants.update(terms_and_constants(x))
    # Constants are case-insensitive for this diagnostic (e.g. Strong/strong).
    # Constants used in formulae are treated as distinct named individuals.
    # This is an explicit finite-domain assumption, not standard unrestricted FOL.
    domain=sorted(constants)+[f'_anon_{i}' for i in range(extra_domain)]
    if not domain:domain=['_anon_0']
    cache={}
    premises=And(*(ground(n,{},domain,cache) for n in formulas))
    q=ground(query,{},domain,cache)
    if len(cache)>max_ground_atoms:
        return {'prediction':None,'status':'too_many_atoms','atoms':len(cache),'seconds':time.perf_counter()-start}
    consistent=satisfiable(premises,all_models=False)
    if consistent is False:
        return {'prediction':None,'status':'inconsistent_premises','atoms':len(cache),'seconds':time.perf_counter()-start}
    if satisfiable(And(premises,Not(q)),all_models=False) is False:prediction='True'
    elif satisfiable(And(premises,q),all_models=False) is False:prediction='False'
    else:prediction='Uncertain'
    return {'prediction':prediction,'status':('bounded_finite_domain_repaired' if repairs else 'bounded_finite_domain'),'repairs':repairs,'atoms':len(cache),'domain_size':len(domain),'seconds':time.perf_counter()-start}

def evaluate_folio_logic(rows,limit=None, extra_domain=1):
    import pandas as pd
    records=[]
    for i,row in enumerate(rows):
        if limit is not None and i>=limit:break
        gold=str(row.get('label','')).strip().capitalize()
        try:r=solve_folio_logic(row,extra_domain=extra_domain)
        except Exception as e:r={'prediction':None,'status':f'{type(e).__name__}: {e}','seconds':0}
        records.append({'index':i,'id':row.get('example_id',row.get('story_id',i)),
                        'gold':gold,'prediction':r['prediction'],'correct':r['prediction']==gold,
                        'status':r['status'],'seconds':r['seconds']})
    df=pd.DataFrame(records)
    supported=df['prediction'].notna()
    print('Examples:',len(df),'Coverage:',f'{supported.mean():.2%}',
          'Overall accuracy:',f'{df.correct.mean():.2%}')
    if supported.any():print('Supported accuracy:',f'{df.loc[supported,"correct"].mean():.2%}')
    print('Status counts:\n',df.status.value_counts().head(12))
    print('Inference seconds:',round(df.seconds.sum(),3))
    return df


In [7]:

import time


def solve_folio_z3(row, timeout_ms=1400, max_memory_mb=512):
    import z3
    start = time.perf_counter()
    premises_raw = split_premises(row['premises-FOL'])
    parsed = [parse_formula_repaired(s) for s in premises_raw]
    query, qr = parse_formula_repaired(row['conclusion-FOL'])
    nodes = [x[0] for x in parsed]
    check_symbol_signatures(nodes + [query])
    repairs = [x[1] for x in parsed if x[1]] + ([qr] if qr else [])

    U = z3.DeclareSort('Individuals')
    constants, predicates = {}, {}
    free_variables = set()

    def term(name, env):
        # Only lexically bound variables are Z3 variables; other terms are constants.
        if name in env:
            return env[name]
        key = name.casefold()
        if key not in constants:
            constants[key] = z3.Const('c_' + key.encode('utf-8').hex(), U)
        return constants[key]

    def convert(n, env):
        op = n[0]
        if op == 'bool':
            return z3.BoolVal(n[1])
        if op == 'eq':
            a, b = term(n[1], env), term(n[2], env)
            return a != b if n[3] else a == b
        if op == 'pred':
            name, args = n[1], n[2]
            key = (name, len(args))
            if key not in predicates:
                predicates[key] = z3.Function(
                    'p_' + name.encode('utf-8').hex(),
                    *([U] * len(args)), z3.BoolSort()
                ) if args else z3.Bool('p_' + name.encode('utf-8').hex())
            if not args:
                return predicates[key]
            return predicates[key](*[term(a, env) for a in args])
        if op == 'not':
            return z3.Not(convert(n[1], env))
        if op in ('forall', 'exists'):
            var = z3.Const('v_' + n[1].encode('utf-8').hex() + '_' + str(len(env)), U)
            child = convert(n[2], {**env, n[1]: var})
            return z3.ForAll([var], child) if op == 'forall' else z3.Exists([var], child)
        a, b = convert(n[1], env), convert(n[2], env)
        return {
            'and': lambda: z3.And(a, b),
            'or': lambda: z3.Or(a, b),
            'xor': lambda: z3.Xor(a, b),
            'implies': lambda: z3.Implies(a, b),
            'iff': lambda: a == b,
        }[op]()

    conditions = [convert(n, {}) for n in nodes]
    q = convert(query, {})

    solver = z3.Solver()
    solver.set(timeout=int(timeout_ms))
    solver.set('max_memory', int(max_memory_mb))
    solver.add(*conditions)

    def check_with(expr=None):
        solver.push()
        if expr is not None:
            solver.add(expr)
        result = solver.check()
        reason = solver.reason_unknown() if result == z3.unknown else None
        solver.pop()
        return result, reason

    consistent, reason = check_with()
    if consistent == z3.unknown:
        return {'prediction':None, 'status':'unknown_premises:' + str(reason), 'seconds':time.perf_counter()-start}
    if consistent == z3.unsat:
        return {'prediction':None, 'status':'inconsistent_premises', 'seconds':time.perf_counter()-start}

    negated, neg_reason = check_with(z3.Not(q))
    if negated == z3.unsat:
        prediction, status = 'True', 'z3_entailed'
    elif negated == z3.unknown:
        prediction, status = None, 'unknown_negated_query:' + str(neg_reason)
    else:
        positive, pos_reason = check_with(q)
        if positive == z3.unsat:
            prediction, status = 'False', 'z3_refuted'
        elif positive == z3.unknown:
            prediction, status = None, 'unknown_positive_query:' + str(pos_reason)
        else:
            prediction, status = 'Uncertain', 'z3_independent'
    if repairs and prediction is not None:
        status += '_repaired'
    return {'prediction':prediction, 'status':status, 'repairs':repairs,
            'constant_count':len(constants), 'predicate_count':len(predicates),
            'seconds':time.perf_counter()-start}


In [8]:
def solve_folio_z3_safe(row, timeout=5, memory_mb=1400, z3_timeout_ms=1500):
    """Run the quantified Z3 checks in this notebook.

    Z3 has its own time and memory limits. Unlike the earlier module-based
    version, this call does not create an isolated Python process.
    """
    return solve_folio_z3(row, timeout_ms=z3_timeout_ms,
                          max_memory_mb=memory_mb)


In [9]:

import json
import re
import time
from pathlib import Path


SYSTEM_PROMPT = """You translate natural-language reasoning problems into first-order logic (FOL).
You are a translator ONLY. NEVER answer the question or predict its truth.
Return exactly one JSON object, with keys "premises" (array of FOL formula strings) and "conclusion" (one FOL formula string). No markdown.
FOL syntax: ∀x (P(x) → Q(x)), ∃x (P(x) ∧ Q(x)), ¬P(a), ∧, ∨, →, ↔, ⊕, =.
Use one formula per premise sentence. Consistently reuse predicate names, constants, and argument order.
Use only simple alphanumeric or underscore identifiers starting with a letter: e.g. Person(alice), Studies(alice), LivesIn(alice, london).
Universal statements require ∀; 'some'/'at least one' require ∃; 'no' means ∀x (P(x) → ¬Q(x)).
All quantified variables must be bound. Represent names as lowercase constants and predicate names as PascalCase.
Use parentheses generously to make quantifier scope and connective precedence explicit.
Do not add world knowledge, implications, or facts not stated in the natural-language input.
Do not invent a yes/no/unknown label. Translate the conclusion even if it is false or unknown."""


def build_prompt(row):
    premises = row.get('premises')
    if isinstance(premises, str):
        text = premises.strip()
    elif isinstance(premises, (list, tuple)):
        text = '\n'.join(f'{i+1}. {p}' for i, p in enumerate(premises))
    else:
        raise ValueError('Missing natural-language premises')
    conclusion = row.get('conclusion')
    if not isinstance(conclusion, str) or not conclusion.strip():
        raise ValueError('Missing natural-language conclusion')
    return f'PREMISES:\n{text}\n\nCONCLUSION:\n{conclusion.strip()}\n\nOutput JSON only.'


def _extract_object(output):
    """Find the first well-formed JSON object without trusting surrounding model text."""
    output = re.sub(r'^\s*```(?:json)?\s*|\s*```\s*$', '', output.strip(), flags=re.I)
    decoder = json.JSONDecoder()
    for i, char in enumerate(output):
        if char == '{':
            try:
                obj, _ = decoder.raw_decode(output[i:])
                if isinstance(obj, dict):
                    return obj
            except json.JSONDecodeError:
                pass
    raise ValueError('No valid JSON object in translator response')


def _has_free_variables(ast, bound=frozenset()):
    """Identify only conventional variable-like names (x,y,z etc) outside binders."""
    op = ast[0]
    if op == 'pred':
        return {t for t in ast[2] if t in {'x','y','z','u','v','w'} and t not in bound}
    if op == 'eq':
        return {t for t in ast[1:3] if t in {'x','y','z','u','v','w'} and t not in bound}
    if op in ('forall','exists'):
        return _has_free_variables(ast[2], bound | {ast[1]})
    if op == 'not':
        return _has_free_variables(ast[1], bound)
    if op == 'bool':
        return set()
    return _has_free_variables(ast[1], bound) | _has_free_variables(ast[2], bound)


def validate_translation(raw):
    obj = _extract_object(raw) if isinstance(raw, str) else raw
    if set(obj) != {'premises', 'conclusion'}:
        raise ValueError('JSON must have only premises and conclusion keys')
    premises = obj['premises']
    conclusion = obj['conclusion']
    if not isinstance(premises, list) or not premises or not all(isinstance(x,str) and x.strip() for x in premises):
        raise ValueError('premises must be a nonempty list of formulas')
    if not isinstance(conclusion,str) or not conclusion.strip():
        raise ValueError('conclusion must be a formula string')
    if len(premises) > 100 or sum(map(len,premises)) + len(conclusion) > 20000:
        raise ValueError('Translation exceeds resource guardrails')
    asts = [parse_formula(s.strip()) for s in premises] + [parse_formula(conclusion.strip())]
    check_symbol_signatures(asts)
    free = set().union(*[_has_free_variables(x) for x in asts])
    if free:
        raise ValueError('Free conventional variables: '+str(sorted(free)))
    return {'premises':[s.strip() for s in premises], 'conclusion':conclusion.strip()}


class QwenFOLTranslator:
    def __init__(self, model_id='Qwen/Qwen2.5-3B-Instruct', device_map='auto', max_new_tokens=900):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        import torch
        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=dtype, device_map=device_map,
            trust_remote_code=False, low_cpu_mem_usage=True
        )
        self.model.eval()
        self.max_new_tokens = int(max_new_tokens)

    def generate(self, row, retry=False):
        messages = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':build_prompt(row)}
        ]
        if retry:
            messages[1]['content'] += '\nIMPORTANT: The prior output failed syntax validation. Return only strictly valid JSON, and explicitly parenthesize all quantifier bodies.'
        tokens = self.tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt')
        # Avoid attention overflow or excessive input length.
        maximum = getattr(self.model.config, 'max_position_embeddings', 32768)
        if tokens.shape[-1] + self.max_new_tokens > maximum:
            raise ValueError(f'Prompt too long: {tokens.shape[-1]} tokens')
        device = self.model.get_input_embeddings().weight.device
        tokens = tokens.to(device)
        with self.torch.inference_mode():
            response = self.model.generate(
                input_ids=tokens,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                use_cache=True
            )
        decoded = self.tokenizer.decode(response[0, tokens.shape[-1]:], skip_special_tokens=True)
        return decoded

    def translate(self, row):
        last_error = None
        for retry in (False, True):
            raw = self.generate(row, retry=retry)
            try:
                return validate_translation(raw)
            except Exception as exc:
                last_error = exc
        raise ValueError(f'Translation invalid after retry: {last_error}')


def infer_from_translation(formulas, timeout=5, memory_mb=1400, z3_timeout_ms=1500):
    """Calls the isolated V5 Z3 worker. Cannot use reference FOL by construction."""
    validated = validate_translation(formulas)
    formal_row = {'premises-FOL':'\n'.join(validated['premises']), 'conclusion-FOL': validated['conclusion']}
    return solve_folio_z3_safe(formal_row, timeout=timeout, memory_mb=memory_mb, z3_timeout_ms=z3_timeout_ms)


def predict_folio_nl(row, translator, **solver_options):
    start = time.perf_counter()
    try:
        formulas = translator.translate(row)
    except Exception as exc:
        return {'prediction':None, 'status':'translation_error:'+str(exc)[:180], 'seconds':time.perf_counter()-start, 'formulas':None}
    result = infer_from_translation(formulas, **solver_options)
    return {**result, 'formulas':formulas, 'total_seconds':time.perf_counter()-start}


def evaluate_folio_nl(rows, translator, limit=10, save_path='folio_nl_predictions.jsonl', **solver_options):
    """Resume-safe cache using index and example ID; labels are used only for evaluation."""
    import pandas as pd
    path = Path(save_path)
    previous = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rec=json.loads(line)
                previous[(rec['index'],str(rec['id']))]=rec
            except (ValueError, KeyError):
                pass
    records = []
    with path.open('a',encoding='utf-8') as fp:
        for index,row in enumerate(rows):
            if index >= limit:break
            ex_id=str(row.get('example_id',index))
            key=(index,ex_id)
            if key in previous:
                rec=previous[key]
            else:
                out=predict_folio_nl(row, translator, **solver_options)
                rec={'index':index,'id':ex_id,'prediction':out.get('prediction'),
                     'status':out.get('status'),'seconds':out.get('total_seconds',out.get('seconds',0)),
                     'formulas':out.get('formulas')}
                fp.write(json.dumps(rec,ensure_ascii=False)+'\n'); fp.flush()
            # Gold annotation NEVER enters model prompt or solver.
            gold=str(row.get('label','')).strip().capitalize()
            records.append({**rec, 'gold':gold, 'correct':rec['prediction']==gold})
            if (index+1)%5 == 0:
                print(f'Completed {index+1}/{limit}',flush=True)
    df=pd.DataFrame(records)
    if len(df):
        supported=df.prediction.notna()
        print(f'Examples={len(df)}, coverage={supported.mean():.2%}, overall accuracy={df.correct.mean():.2%}')
        if supported.any():print(f'Supported accuracy={df.loc[supported,"correct"].mean():.2%}')
        print(df.status.value_counts().head(12))
    return df


In [10]:

from __future__ import annotations

import hashlib
import json
import re
from dataclasses import dataclass
from typing import Any



def _sentences(row):
    p = row['premises']
    if isinstance(p, str):
        return [s.strip() for s in p.splitlines() if s.strip()]
    return [str(s).strip() for s in p if str(s).strip()]


def _clean(s):
    return re.sub(r'\s+', ' ', str(s).strip()).casefold()


def _op(n):
    return n[0]


def _pred_names(node):
    if node[0]=='pred':
        return {node[1].casefold()}
    if node[0] in {'forall','exists'}:
        return _pred_names(node[2])
    if node[0]=='not':
        return _pred_names(node[1])
    if node[0] in {'bool','eq'}:
        return set()
    return _pred_names(node[1]) | _pred_names(node[2])


def _contains_op(node, op):
    if node[0] == op:
        return True
    if node[0] in {'forall','exists'}:
        return _contains_op(node[2], op)
    if node[0]=='not':
        return _contains_op(node[1], op)
    if node[0] in {'pred','eq','bool'}:
        return False
    return _contains_op(node[1], op) or _contains_op(node[2], op)


def _top_node(n):
    while n[0] in {'forall','exists'}:
        n=n[2]
    return n


def _named_atom_matches(nl, ast):
    """Narrow check: affirmative 'NAME is (a/an) X' cannot become ¬X(NAME).

    It intentionally does not attempt to understand complex nested negations.
    """
    nl = nl.strip().rstrip('.!? ')
    match = re.fullmatch(r'([A-Z][a-zA-Z]+)\s+is\s+(?:an?\s+)?(.+)',nl)
    if not match:
        return False
    claim = match.group(2).lower()
    if re.search(r'\b(?:not|neither|never|no)\b',claim):
        return False
    if re.search(r'\b(?:if|or|and|only|either|neither)\b',claim):
        return False
    if ast[0] != 'not':
        return False
    inner=ast[1]
    return inner[0]=='pred' and len(inner[2])==1 and inner[2][0].casefold()==match.group(1).casefold()


def _bare_term_conflicts(asts):
    """Warn if a zero-arity symbol is also used as an individual constant.

    A formula `Rockie ∧ P(Rockie)` is suspicious; do not repair it here.
    """
    props, constants = set(), set()
    def walk(n):
        op=n[0]
        if op=='pred':
            if len(n[2])==0: props.add(n[1].casefold())
            constants.update(x.casefold() for x in n[2])
        elif op=='eq': constants.update((n[1].casefold(),n[2].casefold()))
        elif op in {'forall','exists'}: walk(n[2])
        elif op=='not': walk(n[1])
        elif op!='bool': walk(n[1]);walk(n[2])
    for n in asts:walk(n)
    return props & constants


def _reversed_types(nl, ast):
    """Detect just the syntactic reversal of an exhaustive type enumeration.

    NL says 'There are [N] types of [thing]: ...'. Bad translation often has
    (T1(x) or T2(x)) -> Parent(x) rather than Parent(x) -> (T1(x) or T2(x)).
    """
    if not re.search(r'\b(?:types|kinds|categories)\s+of\b',nl,re.I):
        return False
    n=_top_node(ast)
    return (n[0]=='implies' and n[1][0]=='or' and n[2][0]=='pred'
            and len(n[2][2])==1)


def audit_translation(row, formulas):
    """Return list of human-readable hard verification errors. No labels used."""
    sentences = _sentences(row)
    valid=validate_translation(formulas)
    errors=[]
    if len(valid['premises'])!=len(sentences):
        errors.append(f'premise_count: expected {len(sentences)}, got {len(valid["premises"])}')
        return errors
    premises=[parse_formula(s) for s in valid['premises']]
    query=parse_formula(valid['conclusion'])
    for i,(nl,ast) in enumerate(zip(sentences,premises),1):
        if _reversed_types(nl,ast):
            errors.append(f'premise_{i}: category enumeration implication reversed')
        if re.search(r'\bneither\b.*\bnor\b',nl,re.I):
            # "Neither A nor B" requires conjunction of negations, unless
            # source mixes it into another connective.
            if ast[0]=='or' and not re.search(r'\b(?:if|either)\b',nl,re.I):
                errors.append(f'premise_{i}: neither/nor rendered as disjunction')
        if _named_atom_matches(nl,ast):
            errors.append(f'premise_{i}: affirmative named fact negated')
    nlq=row['conclusion']
    if _named_atom_matches(nlq,query):
        errors.append('conclusion: affirmative named-entity claim negated')
    if (_contains_op(query,'xor') and
        not re.search(r'\b(?:exactly one|one but not both|exclusive(?:ly)?|but not both)\b',nlq,re.I)):
        errors.append('conclusion: XOR used without explicit exclusive meaning')
    # Do not apply this to all premises, where a genuine zero-arity flag may
    # have been intentionally introduced by the translator.
    bad=_bare_term_conflicts(premises+[query])
    if bad:
        errors.append('bare symbols also used as constants: '+', '.join(sorted(bad)))
    return errors


class VerifiedFOLTranslator:
    """Wraps the existing 7B translator; no model loading.

    1) base.translate(row) first; 2) reject evident semantic errors;
    3) retry using base.generate with natural-language guardrails;
    4) freeze premises for identical source text across different questions.
    A cached premise set is discarded if it fails symbol-signature checks
    with a new conclusion; it is never silently mutated.
    """

    def __init__(self, base, retries=2, cache_path=None):
        self.base=base
        self.retries=int(retries)
        self.premise_cache={}
        self.last_trace=[]
        self.cache_path=cache_path
        if cache_path:
            try:
                with open(cache_path,encoding='utf8') as fh:
                    for line in fh:
                        item=json.loads(line)
                        if item.get('version')==1:
                            self.premise_cache[item['key']]=item['premises']
            except FileNotFoundError:
                pass

    def _key(self,row):
        # Normalize whitespace only; don't case-fold entities or predicate names
        src='\n'.join(_clean(s) for s in _sentences(row))
        return hashlib.sha256(src.encode()).hexdigest()

    def _store(self,key,premises):
        if key not in self.premise_cache:
            self.premise_cache[key]=list(premises)
            if self.cache_path:
                with open(self.cache_path,'a',encoding='utf8') as fh:
                    fh.write(json.dumps({'version':1,'key':key,'premises':premises},ensure_ascii=False)+'\n')

    def _candidate(self,row, attempt):
        if attempt==0:
            return self.base.translate(row)
        # Retry with a *request-specific* but label-free corrective prompt.
        # `generate()` is the working user's implementation (single-GPU FP32).
        p=_sentences(row)
        extra=(
            '\n\nTRANSLATION VALIDATION: Return exactly '+str(len(p))+
            ' premise formulas, in the identical order. Preserve the original '
            'conclusion polarity, inclusive OR, named constants and implication '
            'direction. A sentence enumerating all types must have the '
            'parent category imply the disjunction of types. Never translate '
            'the expected answer. Output strictly valid JSON.'
        )
        newrow={'premises':row['premises'], 'conclusion':row['conclusion']+extra}
        raw=self.base.generate(newrow,retry=True)
        return validate_translation(raw)

    def translate(self,row):
        self.last_trace=[]
        k=self._key(row)
        errors=[]
        for attempt in range(self.retries+1):
            try:
                cand=self._candidate(row,attempt)
                cand=validate_translation(cand)
                # Strictly require per-premise coverage.
                if len(cand['premises'])!=len(_sentences(row)):
                    raise ValueError('premise count mismatch')
                # Prefer cached premises when available to guarantee common KB.
                if k in self.premise_cache:
                    combined={'premises':self.premise_cache[k],
                              'conclusion':cand['conclusion']}
                    try:
                        validate_translation(combined)
                        cand=combined
                    except ValueError:
                        # Incompatible symbol signatures: no unsafe repair.
                        raise ValueError('cached premise signatures conflict with conclusion')
                flags=audit_translation(row,cand)
                if flags:
                    raise ValueError('; '.join(flags))
                cache_hit = (k in self.premise_cache)
                self._store(k,cand['premises'])
                self.last_trace.append({'attempt':attempt+1,'accepted':True,
                                        'premise_cache_hit':cache_hit})
                return cand
            except Exception as exc:
                msg=f'{type(exc).__name__}: {str(exc)[:300]}'
                self.last_trace.append({'attempt':attempt+1,'accepted':False,'error':msg})
                errors.append(msg)
        raise ValueError('verification failed: '+' | '.join(errors)[-480:])


def predict_verified_folio(row, translator, **solver_options):
    """The usual Z3 route, with no access to row['label']."""
    import time
    start=time.perf_counter()
    try:
        formulas=translator.translate(row)
    except Exception as exc:
        return {'prediction':None,'status':'translation_error:'+str(exc)[:250],
                'formulas':None,'seconds':time.perf_counter()-start}
    result=infer_from_translation(formulas, **solver_options)
    return {**result, 'formulas':formulas,'total_seconds':time.perf_counter()-start}


def evaluate_verified_folio(rows, translator, limit=20, save_path='folio_verified_7b_test20.jsonl', **solver_options):
    """Use existing resumable scorer; labels used only in scoring."""
    return evaluate_folio_nl(rows,translator,limit=limit,save_path=save_path,**solver_options)


In [11]:
"""Unified interface for NL -> logical representation -> classical inference.

No reference annotations, gold labels, or benchmark IDs enter prediction.
Explicit mode selects semantics; mode='auto' routes by input FIELD STRUCTURE
or defaults to classical FOL for generic facts and queries.
"""
from __future__ import annotations
validate_ast = validate_ast_translation
validate_fol = validate_translation
import csv
import json
import os
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path




class SingleGPUFOLTranslator:
    """Lazy-load FP32 Qwen2.5-7B onto a *single* GPU; no auto-sharding.

    Important: 7B FP32 typically needs >30GB free VRAM. A CUDA GPU with enough memory is required for full-precision inference.
    """
    def __init__(self, model_id='Qwen/Qwen2.5-7B-Instruct', max_new_tokens=1400,
                 model=None, tokenizer=None, device='cuda:0'):
        self.model_id=model_id
        self.max_new_tokens=max_new_tokens
        self.model=model
        self.tokenizer=tokenizer
        self.device=device

    def _ensure_model(self):
        if self.model is not None and self.tokenizer is not None:
            return
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        if not torch.cuda.is_available():
            raise RuntimeError('FOLIO 7B FP32 requires a CUDA GPU; no GPU detected')
        self.tokenizer=AutoTokenizer.from_pretrained(self.model_id)
        self.model=AutoModelForCausalLM.from_pretrained(
            self.model_id, dtype=torch.float32, attn_implementation='eager',
            device_map=self.device, trust_remote_code=False)
        self.model.eval()

    def generate(self,row,retry=False):
        import torch
        self._ensure_model()
        prompt=build_prompt(row)
        if retry:
            prompt+='\nYour previous attempt was invalid. Keep exactly one FOL formula per premise. Use only permitted FOL operators; return valid JSON.'
        messages=[{'role':'system','content':SYSTEM_PROMPT+'\nPreserve the original conclusion polarity and implication directions. Never skip a premise.'},
                  {'role':'user','content':prompt}]
        encoded=self.tokenizer.apply_chat_template(messages,tokenize=True,
            add_generation_prompt=True,return_tensors='pt',return_dict=True)
        if isinstance(encoded,torch.Tensor):
            encoded={'input_ids':encoded,'attention_mask':torch.ones_like(encoded)}
        input_ids=encoded['input_ids']
        attention_mask=encoded.get('attention_mask')
        dev=self.model.get_input_embeddings().weight.device
        input_ids=input_ids.to(dev)
        kw={'input_ids':input_ids}
        if attention_mask is not None:
            kw['attention_mask']=attention_mask.to(dev)
        if input_ids.shape[1]+self.max_new_tokens>getattr(self.model.config,'max_position_embeddings',32768):
            raise ValueError('Prompt plus completion exceeds model context limit')
        with torch.inference_mode():
            result=self.model.generate(**kw,max_new_tokens=self.max_new_tokens,
                do_sample=False,use_cache=False,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id)
        return self.tokenizer.decode(result[0,input_ids.shape[1]:],skip_special_tokens=True)

    def translate(self,row):
        count=len(_sentences(row['premises']))
        errors=[]
        for attempt in range(3):
            try:
                # Number original premises, not the query. One output per sentence.
                numbered='\n'.join(f'{i+1}. {s}' for i,s in enumerate(_sentences(row['premises'])))
                request={'premises':numbered,'conclusion':row['conclusion']}
                raw=self.generate(request,retry=attempt>0)
                formulas=validate_fol(raw)
                if len(formulas['premises'])!=count:
                    raise ValueError(f'Premise count mismatch: expected {count}, got {len(formulas["premises"])}')
                return formulas
            except Exception as exc:
                errors.append(f'{type(exc).__name__}: {exc}')
        raise ValueError('Translation failed: '+' | '.join(errors)[-300:])


def _sentences(v):
    if isinstance(v,(list,tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    if isinstance(v,str):
        lines=[s.strip() for s in v.splitlines() if s.strip()]
        return lines if len(lines)>1 else split_statements(v)
    raise ValueError('Premises/context must be a string or a list')


def normalized_mode(row,mode='auto'):
    if mode!='auto':
        if mode not in ('ruletaker','counterbench','folio'):
            raise ValueError(f'Unknown mode {mode}')
        return mode
    if 'type' in row and 'given_info' in row and 'question' in row:
        return 'counterbench'
    if 'premises' in row and 'conclusion' in row:
        return 'folio'
    # RuleTaker-specific context/question schema, otherwise classical FOL.
    if 'context' in row and 'question' in row:
        return 'ruletaker'
    return 'folio'


def _extract(row,mode):
    if mode=='counterbench':
        return str(row['given_info']),str(row['question'])
    if mode=='ruletaker':
        p=row.get('context',row.get('premises'))
        q=row.get('question',row.get('query',row.get('conclusion')))
        return p,q
    p=row.get('premises',row.get('context',row.get('knowledge_base')))
    q=row.get('conclusion',row.get('query',row.get('question')))
    return p,q


@dataclass
class Prediction:
    id: str
    mode: str
    prediction: str | None
    binary_label: str
    supported: bool
    status: str
    seconds: float
    solver_seconds: float
    trace: object = None


class ReasoningAgent:
    def __init__(self, folio_translator=None, ruletaker_translator=None,
                 use_verification=True, cache_path='.folio_premise_cache.jsonl',
                 z3_timeout_ms=1500, timeout=5, memory_mb=1400):
        self._folio=folio_translator
        self._ruletaker=ruletaker_translator
        self.use_verification=use_verification
        self.cache_path=cache_path
        self.z3_kwargs=dict(timeout=timeout,memory_mb=memory_mb,z3_timeout_ms=z3_timeout_ms)

    def folio_translator(self):
        if self._folio is None:
            raw=SingleGPUFOLTranslator()
            self._folio=VerifiedFOLTranslator(raw,retries=2,cache_path=self.cache_path) if self.use_verification else raw
        return self._folio

    def ruletaker_translator(self):
        if self._ruletaker is None:
            self._ruletaker=LLMTranslator(model='Qwen/Qwen2.5-7B-Instruct',
                device='cuda',precision='fp32',smoke_test=False)
        return self._ruletaker

    def predict_row(self,row,mode='auto',index=0):
        mode=normalized_mode(row,mode)
        p,q=_extract(row,mode)
        identifier=str(row.get('id',row.get('example_id',row.get('question_id',index))))
        start=time.perf_counter()
        prediction=None;status='unsupported';trace=None;solver_seconds=0.0
        try:
            if mode=='counterbench':
                t=time.perf_counter()
                result=infer_counterbench({'type':row['type'],'given_info':p,'question':q})
                solver_seconds=time.perf_counter()-t
                prediction=result['prediction']
                status='boolean_scm'
                trace={'target':result.get('target'),'interventions':result.get('intervention_stages'),
                       'state':result.get('states')}
            elif mode=='ruletaker':
                premises=_sentences(p)
                clauses=[fast_parse(s) for s in premises]
                query=fast_parse(q)
                if all(c is not None for c in clauses) and query is not None:
                    logical={'premises':clauses,'query':query}
                    validate_ast(logical)
                else:
                    logical=self.ruletaker_translator().translate(p,q,'logic')
                    if len(logical['premises'])!=len(premises):
                        raise ValueError('RuleTaker premise count mismatch')
                t=time.perf_counter()
                result=infer_closed_world(logical)
                solver_seconds=time.perf_counter()-t
                prediction=result.label
                status='stratified_rule_inference'
                trace=result.trace
            else:
                form_row={'premises':p,'conclusion':q}
                forms=self.folio_translator().translate(form_row)
                t=time.perf_counter()
                result=infer_from_translation(forms,**self.z3_kwargs)
                solver_seconds=time.perf_counter()-t
                prediction=result.get('prediction')
                status=result.get('status','unknown_solver_status')
                trace={'formulas':forms,'repairs':result.get('repairs',[])}
        except Exception as exc:
            status=f'{type(exc).__name__}: {str(exc)[:240]}'
        binary='True' if prediction=='True' else 'False'
        return Prediction(identifier,mode,prediction,binary,prediction is not None,
                          status,time.perf_counter()-start,solver_seconds,trace)

    def predict(self,premises,conclusion,mode='folio'):
        row={'premises':premises,'conclusion':conclusion}
        return self.predict_row(row,mode=mode).binary_label

    def evaluate(self, rows, mode='auto', output_csv='submission/submission.csv',
                 diagnostics_path='submission/diagnostics.jsonl'):
        """Rows can be public/dev or private (unlabeled). Never reads gold for inference."""
        records=[]
        for index,row in enumerate(rows):
            records.append(self.predict_row(row,mode=mode,index=index))
            if (index+1)%10==0:
                print(f'Completed {index+1} examples',flush=True)
        if not records:
            raise ValueError('No input rows; refusing to produce an empty prediction submission')
        out=Path(output_csv);out.parent.mkdir(parents=True,exist_ok=True)
        with out.open('w',newline='',encoding='utf-8') as fh:
            w=csv.DictWriter(fh,fieldnames=['id','label']);w.writeheader()
            for r in records:w.writerow({'id':r.id,'label':r.binary_label})
        log=Path(diagnostics_path);log.parent.mkdir(parents=True,exist_ok=True)
        with log.open('w',encoding='utf-8') as fh:
            for r in records:fh.write(json.dumps(asdict(r),ensure_ascii=False,default=str)+'\n')
        total=sum(r.seconds for r in records)
        inf=sum(r.solver_seconds for r in records)
        print(f'Rows: {len(records)}; supported: {sum(r.supported for r in records)}/{len(records)}')
        print(f'End-to-end sum: {total:.3f}s; classical inference sum: {inf:.3f}s')
        gold=[]
        for row in rows:
            g=row.get('label',row.get('answer'))
            if g is None: gold.append(None)
            elif isinstance(g,bool):gold.append('True' if g else 'False')
            else:
                s=str(g).lower().strip()
                gold.append('True' if s in ('true','1','yes','entailment')
                    else 'False' if s in ('false','0','no','contradiction','uncertain','unknown','neutral')
                    else None)
        scored=[(r.binary_label,g) for r,g in zip(records,gold) if g is not None]
        if scored:
            acc=sum(p==g for p,g in scored)/len(scored)
            print(f'Public-data binary accuracy: {acc:.2%} ({len(scored)} labeled)')
        print(f'Wrote {out} and {log}')
        return records


def load_rows(path):
    path=Path(path)
    if path.suffix.lower()=='.csv':
        with path.open(encoding='utf-8-sig',newline='') as fh:return list(csv.DictReader(fh))
    if path.suffix.lower()=='.jsonl':
        return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    obj=json.loads(path.read_text(encoding='utf-8'))
    if isinstance(obj,list):return obj
    if isinstance(obj,dict):
        for key in ('data','examples','questions'):
            if isinstance(obj.get(key),list):return obj[key]
    raise ValueError('Expected JSON array, JSONL, CSV, or container with a data/examples/questions list')



In [12]:
# Keep a single copy of Qwen on the GPU for both translation paths.
def _shared_ruletaker_translator(self):
    if self._ruletaker is None:
        self._folio = self.folio_translator()
        base = self._folio.base if hasattr(self._folio, "base") else self._folio
        base._ensure_model()
        parser = LLMTranslator(model=base.model_id, device="cuda",
                               precision="fp32", smoke_test=False)
        parser._model = base.model
        parser._tokenizer = base.tokenizer
        parser._device = "cuda:0"
        self._ruletaker = parser
    return self._ruletaker

ReasoningAgent.ruletaker_translator = _shared_ruletaker_translator


## Evaluation

The public RuleTaker validation split, both CounterBench question files, and the FOLIO validation split are evaluated without row limits. Accuracy and processing time are printed separately for each dataset.


In [13]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download


def load_evaluation_data():
    ruletaker = list(load_dataset('tasksource/ruletaker', split='dev'))
    folio = list(load_dataset('tasksource/folio', split='validation'))
    counterbench = []
    for filename in ('data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json'):
        file_path = hf_hub_download(
            repo_id='CounterBench/CounterBench',
            filename=filename,
            repo_type='dataset'
        )
        counterbench.extend(load_rows(file_path))
    for i, row in enumerate(ruletaker):
        row['id'] = f'ruletaker_{i}'
    for i, row in enumerate(counterbench):
        row['id'] = f'counterbench_{i}'
    for i, row in enumerate(folio):
        row['id'] = f'folio_{row.get("example_id", i)}'
    return {'ruletaker': ruletaker, 'counterbench': counterbench, 'folio': folio}


def expected_label(row, dataset):
    value = row.get('answer') if dataset == 'counterbench' else row.get('label')
    if value is None:
        return None
    value = str(value).strip().lower()
    if value in ('true', 'yes', '1', 'entailment', 'entailed'):
        return 'True'
    if value in ('false', 'no', '0', 'not entailment', 'not_entailment',
                 'contradiction', 'neutral', 'unknown', 'uncertain'):
        return 'False'
    return None


/home1/arti_r/miniconda3/envs/fol_assignment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
agent = ReasoningAgent(use_verification=True, cache_path='folio_premise_cache.jsonl')
datasets = load_evaluation_data()
all_predictions = []
summary = []

for dataset, rows in datasets.items():
    start = time.perf_counter()
    correct = 0
    labeled = 0
    supported = 0
    solver_time = 0.0

    for index, row in enumerate(rows):
        result = agent.predict_row(row, mode=dataset, index=index)
        target = expected_label(row, dataset)
        all_predictions.append({'id': result.id, 'label': result.binary_label})
        supported += int(result.supported)
        solver_time += result.solver_seconds
        if target is not None:
            labeled += 1
            correct += int(result.binary_label == target)
        if (index + 1) % 100 == 0:
            print(f'{dataset}: {index + 1}/{len(rows)}')

    elapsed = time.perf_counter() - start
    result_row = {
        'Dataset': dataset, 'Examples': len(rows),
        'Labeled': labeled,
        'Accuracy (%)': round(100 * correct / labeled, 2) if labeled else None,
        'Coverage (%)': round(100 * supported / len(rows), 2),
        'Total time (s)': round(elapsed, 2),
        'Average time (s)': round(elapsed / len(rows), 3),
        'Solver time (s)': round(solver_time, 2)
    }
    summary.append(result_row)
    print(result_row)

pd.DataFrame(all_predictions).to_csv('submission.csv', index=False)
display(pd.DataFrame(summary))
print('Saved submission.csv')


ruletaker: 100/75872
ruletaker: 200/75872
ruletaker: 300/75872
ruletaker: 400/75872
ruletaker: 500/75872
ruletaker: 600/75872
ruletaker: 700/75872
ruletaker: 800/75872
ruletaker: 900/75872
ruletaker: 1000/75872
ruletaker: 1100/75872
ruletaker: 1200/75872
ruletaker: 1300/75872
ruletaker: 1400/75872
ruletaker: 1500/75872
ruletaker: 1600/75872
ruletaker: 1700/75872
ruletaker: 1800/75872
ruletaker: 1900/75872
ruletaker: 2000/75872
ruletaker: 2100/75872
ruletaker: 2200/75872
ruletaker: 2300/75872
ruletaker: 2400/75872
ruletaker: 2500/75872
ruletaker: 2600/75872
ruletaker: 2700/75872
ruletaker: 2800/75872
ruletaker: 2900/75872
ruletaker: 3000/75872
ruletaker: 3100/75872
ruletaker: 3200/75872
ruletaker: 3300/75872
ruletaker: 3400/75872
ruletaker: 3500/75872
ruletaker: 3600/75872
ruletaker: 3700/75872
ruletaker: 3800/75872
ruletaker: 3900/75872
ruletaker: 4000/75872
ruletaker: 4100/75872
ruletaker: 4200/75872
ruletaker: 4300/75872
ruletaker: 4400/75872
ruletaker: 4500/75872
ruletaker: 4600/758

,Dataset,Examples,Labeled,Accuracy (%),Coverage (%),Total time (s),Average time (s),Solver time (s)
0,ruletaker,75872,75872,97.97,95.83,26.15,0.0,12.79
1,counterbench,1200,1200,88.75,83.33,0.08,0.0,0.07
2,folio,203,203,64.53,0.00,0.00,0.0,0.00


Saved submission.csv
